# ❤️ Proyecto: Predicción de Enfermedades Cardiovasculares

## 🎯 Objetivos del Proyecto

En este proyecto práctico aprenderás a:

1. **Trabajar con datos médicos reales** del UCI Heart Disease Dataset
2. **Construir un pipeline completo** de Machine Learning
3. **Evaluar modelos de clasificación binaria** con múltiples métricas
4. **Comparar diferentes algoritmos** (SGD vs Random Forest)
5. **Registrar experimentos** con MLflow de forma profesional
6. **Interpretar resultados médicos** con responsabilidad

---

## ❤️ ¿Por qué es importante?

Las **Enfermedades Cardiovasculares (ECV)** son la principal causa de muerte en el mundo:

- 💔 Causan **17.9 millones** de muertes al año
- ⚠️ **90% son prevenibles** con detección temprana
- 🏥 Un diagnóstico temprano puede **salvar vidas**
- 🤖 La IA puede ayudar a **detectar patrones** que salven vidas

---

## 📊 El Dataset: UCI Heart Disease

- **303 pacientes** con datos clínicos
- **14 características** médicas (edad, presión arterial, colesterol, etc.)
- **Variable objetivo**: Presencia de enfermedad cardíaca (0/1)
- **Tipo de problema**: Clasificación binaria

---

## 🎓 Ejercicio Especial: Completa los Comentarios

**🚨 IMPORTANTE**: A lo largo de este notebook verás comentarios como:

```python
# TODO: [Completa aquí] ¿Qué hace esta función?
```

**Tu misión** es completar estos comentarios explicando:
- ¿Qué hace el código?
- ¿Por qué es importante?
- ¿Qué resultado esperamos?

Esto te ayudará a:
- ✅ Entender profundamente cada paso
- ✅ Practicar documentación de código
- ✅ Prepararte para proyectos reales

---

## 🚀 ¡Empecemos!

## ⚙️ Paso 2: Configuración de MLflow

Configuramos el experimento donde registraremos todos nuestros entrenamientos.

## 📚 Paso 3: Importar Librerías

Importamos todas las herramientas necesarias para el proyecto.

In [0]:
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

# TODO: [Completa aquí] ¿Qué debes hacer con esta variable email?
email = ''  # ⚠️ CAMBIAR POR TU EMAIL DE DATABRICKS

# Validación
if not email:
    print("⚠️  ADVERTENCIA: Debes configurar tu email antes de continuar")
else:
    # TODO: [Completa aquí] ¿Para qué sirve set_tracking_uri?
    mlflow.set_tracking_uri("databricks")
    
    # TODO: [Completa aquí] ¿Qué hace set_experiment?
    experiment_name = f"/Users/{email}/5-prediccion-infarto"
    mlflow.set_experiment(experiment_name)
    
    print("=" * 70)
    print("✅ MLflow configurado correctamente")
    print("=" * 70)
    print(f"📊 Experimento: {experiment_name}")
    print(f"❤️  Proyecto: Predicción de Enfermedades Cardiovasculares")
    print("=" * 70)

In [0]:
# Librerías básicas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# TODO: [Completa aquí] ¿Para qué sirve train_test_split?
from sklearn.model_selection import train_test_split

# TODO: [Completa aquí] ¿Qué es un Pipeline en sklearn?
from sklearn.pipeline import Pipeline

# Preprocesamiento
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# TODO: [Completa aquí] ¿Qué modelos vamos a usar?
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier

# TODO: [Completa aquí] ¿Para qué sirve cross_val_score?
from sklearn.model_selection import cross_val_score, cross_val_predict

# TODO: [Completa aquí] ¿Qué métricas usaremos y por qué?
from sklearn.metrics import (
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix, 
    ConfusionMatrixDisplay, 
    roc_auc_score,
    accuracy_score,
    classification_report,
    roc_curve
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Todas las librerías importadas correctamente")

## ❤️ Contexto del Proyecto

### El Problema

En este proyecto trabajaremos con el **UCI Heart Disease Dataset**, uno de los datasets más importantes en medicina predictiva. 

### 📊 Datos Clave sobre ECV

- 💔 **Principal causa de muerte** a nivel mundial
- 📈 **17.9 millones** de muertes anuales
- ⚠️ **90% son prevenibles** con detección temprana
- 🎯 **Diagnóstico temprano** = vidas salvadas
- 🤖 **IA** puede detectar patrones imperceptibles para humanos

### 🎯 Nuestro Objetivo

Construir un modelo de Machine Learning que pueda **predecir la presencia de enfermedad cardiovascular** basándose en características clínicas del paciente.

### ⚕️ Responsabilidad Ética

> ⚠️ **IMPORTANTE**: Este es un proyecto educativo. Los modelos médicos reales requieren:
> - Validación clínica exhaustiva
> - Aprobación regulatoria
> - Supervisión médica profesional
> - Consideraciones éticas y legales

Nuestro objetivo es aprender las técnicas, no reemplazar el criterio médico.

## 📚 Fundamentos: Clasificación Binaria

### 🎯 ¿Qué es Clasificación Binaria?

Un problema donde el modelo debe elegir entre **dos clases**:
- ✅ Clase Positiva (1): Tiene enfermedad cardíaca
- ❌ Clase Negativa (0): No tiene enfermedad cardíaca

---

### 🔄 Pipeline de Entrenamiento

```
1. Preparación de Datos
   ↓
2. Selección del Modelo
   ↓
3. Entrenamiento
   ↓
4. Ajuste de Hiperparámetros
   ↓
5. Evaluación
```

---

### 📊 Métricas de Evaluación

#### 1. **Matriz de Confusión**

|                    | Predicción: No (0) | Predicción: Sí (1) |
|--------------------|--------------------|--------------------|
| **Real: No (0)**   | TN (Verdadero Neg) | FP (Falso Positivo)|
| **Real: Sí (1)**   | FN (Falso Negativo)| TP (Verdadero Pos) |

#### 2. **Precision (Precisión)**
```
Precision = TP / (TP + FP)
```
- "De los que predije como enfermos, ¿cuántos realmente lo están?"
- **Importante cuando**: Los falsos positivos son costosos

#### 3. **Recall (Sensibilidad)**
```
Recall = TP / (TP + FN)
```
- "De todos los enfermos reales, ¿cuántos detecté?"
- **Importante cuando**: No podemos perder ningún caso positivo (medicina)

#### 4. **F1-Score**
```
F1 = 2 × (Precision × Recall) / (Precision + Recall)
```
- Balance entre Precision y Recall
- Útil cuando las clases están desbalanceadas

#### 5. **ROC-AUC**
- Curva ROC: Representa el trade-off entre True Positive Rate y False Positive Rate
- AUC: Área bajo la curva (0.5 = aleatorio, 1.0 = perfecto)

#### 6. **Validación Cruzada**
- Divide datos en K partes (folds)
- Entrena K veces, cada vez con un fold diferente como test
- Promedia resultados para obtener estimación robusta

---

### 🎯 ¿Qué métrica usar?

| Situación | Métrica Principal |
|-----------|-------------------|
| Clases balanceadas | Accuracy |
| No perder positivos (medicina) | **Recall** ⭐ |
| Evitar falsos positivos | Precision |
| Balance general | F1-Score |
| Comparar modelos | ROC-AUC |

**En medicina, típicamente priorizamos Recall** porque es mejor detectar un caso falso positivo que perder un verdadero positivo.

## 📁 Paso 4: Cargar y Explorar los Datos

Cargamos el dataset y realizamos un análisis exploratorio inicial.

In [0]:
# TODO: [Completa aquí] ¿De dónde cargamos los datos y qué contiene este archivo?
data = pd.read_csv('../data/raw/heart.csv')

print("=" * 70)
print("❤️  DATASET CARGADO: UCI HEART DISEASE")
print("=" * 70)
print(f"📊 Número de muestras: {len(data)}")
print(f"📈 Número de características: {len(data.columns) - 1}")  # -1 porque target no es característica
print(f"🎯 Variable objetivo: target (0 = No enfermedad, 1 = Enfermedad)")
print("=" * 70)

In [0]:
# TODO: [Completa aquí] ¿Qué nos muestra head() y por qué es útil?
print("🔍 Primeras 5 filas del dataset:\n")
data.head()

### 📋 Diccionario de Datos

Cada columna representa una característica médica importante:

#### 👤 Datos Demográficos
1. **age**: Edad del paciente en años
2. **sex**: Sexo (1 = masculino, 0 = femenino)

#### 💊 Síntomas y Diagnóstico
3. **cp**: Tipo de dolor de pecho (Chest Pain)
   - 0: Angina típica
   - 1: Angina atípica
   - 2: Dolor no anginoso
   - 3: Asintomático

4. **exang**: Angina inducida por ejercicio (1 = sí, 0 = no)

#### 🩺 Mediciones Clínicas
5. **trestbps**: Presión arterial en reposo (mm Hg)
6. **chol**: Colesterol sérico (mg/dl)
7. **fbs**: Azúcar en sangre en ayunas > 120 mg/dl (1 = verdadero, 0 = falso)
8. **thalach**: Frecuencia cardíaca máxima alcanzada

#### 📊 Resultados de Pruebas
9. **restecg**: Resultados electrocardiográficos en reposo
   - 0: Normal
   - 1: Anormalidad de onda ST-T
   - 2: Hipertrofia ventricular izquierda

10. **oldpeak**: Depresión del segmento ST inducida por ejercicio

11. **slope**: Pendiente del segmento ST durante ejercicio
    - 0: Ascendente
    - 1: Plano
    - 2: Descendente

12. **ca**: Número de vasos principales coloreados por fluoroscopia (0-3)

13. **thal**: Resultados de prueba de talasemia
    - 1: Normal
    - 2: Defecto fijo
    - 3: Defecto reversible

#### 🎯 Variable Objetivo
14. **target**: Presencia de enfermedad cardíaca
    - 0: No enfermedad
    - 1: Enfermedad presente

---

**💡 Tip**: En medicina, cada una de estas características tiene significado clínico. Un buen data scientist debe entender el dominio del problema.

In [0]:
# TODO: [Completa aquí] ¿Qué información nos da info() sobre el dataset?
print("ℹ️  Información general del dataset:\n")
data.info()

In [0]:
# TODO: [Completa aquí] ¿Qué estadísticas nos proporciona describe()?
print("📊 Estadísticas descriptivas:\n")
data.describe()

### 💡 Observaciones Clave

**TODO: [Completa aquí] Basándote en las estadísticas, ¿qué observaciones puedes hacer sobre:**
- La edad media de los pacientes
- El rango de valores de cada característica
- La presencia de valores nulos
- Características que pueden necesitar normalización

In [0]:
# TODO: [Completa aquí] ¿Por qué es importante analizar la distribución del target?
print("🎯 Distribución de la variable objetivo:\n")
target_counts = data.target.value_counts()
print(target_counts)
print(f"\nProporción:")
print(f"  - Sin enfermedad (0): {target_counts[0]/len(data)*100:.1f}%")
print(f"  - Con enfermedad (1): {target_counts[1]/len(data)*100:.1f}%")

# TODO: [Completa aquí] ¿Está balanceado el dataset? ¿Por qué es importante?
balance_ratio = min(target_counts) / max(target_counts)
if balance_ratio > 0.8:
    print(f"\n✅ Dataset balanceado (ratio: {balance_ratio:.2f})")
else:
    print(f"\n⚠️  Dataset desbalanceado (ratio: {balance_ratio:.2f})")

### 🔍 Análisis Exploratorio Visual

Visualicemos las relaciones entre características para entender mejor los datos.

In [ ]:
# TODO: [Completa aquí] ¿Qué podemos aprender de las visualizaciones?

# Crear figura con múltiples subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('❤️ Análisis Exploratorio de Datos - Heart Disease', fontsize=16, fontweight='bold')

# Gráfico 1: Distribución de edad por target
axes[0, 0].hist([data[data.target==0]['age'], data[data.target==1]['age']], 
                bins=20, label=['Sin enfermedad', 'Con enfermedad'], alpha=0.7)
axes[0, 0].set_xlabel('Edad')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].set_title('Distribución de Edad por Diagnóstico')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Gráfico 2: Colesterol vs Presión Arterial
scatter = axes[0, 1].scatter(data['chol'], data['trestbps'], c=data['target'], 
                             alpha=0.6, cmap='RdYlGn_r', edgecolors='black', linewidth=0.5)
axes[0, 1].set_xlabel('Colesterol (mg/dl)')
axes[0, 1].set_ylabel('Presión Arterial en Reposo (mm Hg)')
axes[0, 1].set_title('Colesterol vs Presión Arterial')
plt.colorbar(scatter, ax=axes[0, 1], label='Target')
axes[0, 1].grid(True, alpha=0.3)

# Gráfico 3: Tipo de dolor de pecho por diagnóstico
cp_target = pd.crosstab(data['cp'], data['target'], normalize='index') * 100
cp_target.plot(kind='bar', ax=axes[1, 0], alpha=0.8)
axes[1, 0].set_xlabel('Tipo de Dolor de Pecho')
axes[1, 0].set_ylabel('Porcentaje (%)')
axes[1, 0].set_title('Tipo de Dolor de Pecho vs Diagnóstico')
axes[1, 0].legend(['Sin enfermedad', 'Con enfermedad'])
axes[1, 0].set_xticklabels(['Típica', 'Atípica', 'No anginoso', 'Asintomático'], rotation=45)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Gráfico 4: Matriz de correlación (top 10 características)
top_features = ['age', 'sex', 'cp', 'trestbps', 'chol', 'thalach', 'exang', 'oldpeak', 'ca', 'target']
corr_matrix = data[top_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            ax=axes[1, 1], cbar_kws={'label': 'Correlación'})
axes[1, 1].set_title('Matriz de Correlación')

plt.tight_layout()
plt.show()

print("✅ Visualizaciones generadas")
print("\n💡 TODO: [Completa aquí] ¿Qué patrones observas en los gráficos?")

## 🔀 Paso 5: Dividir los Datos

Dividimos el dataset en conjuntos de entrenamiento y prueba.

In [0]:
# TODO: [Completa aquí] ¿Por qué dividimos en train y test? ¿Qué es test_size y random_state?
train_set, test_set = train_test_split(data, test_size=0.2, random_state=5)

print("=" * 70)
print("🔀 DIVISIÓN DE DATOS")
print("=" * 70)
print(f"📊 Total de muestras: {len(data)}")
print(f"📚 Conjunto de entrenamiento: {len(train_set)} ({len(train_set)/len(data)*100:.1f}%)")
print(f"🧪 Conjunto de prueba: {len(test_set)} ({len(test_set)/len(data)*100:.1f}%)")
print("=" * 70)
print("✅ Datos divididos correctamente")

In [0]:
# TODO: [Completa aquí] ¿Qué nos muestran los histogramas y por qué son importantes?
print("📊 Generando histogramas de distribuciones...\n")
train_set.hist(bins=50, figsize=(20, 15))
plt.suptitle('Distribución de Características - Conjunto de Entrenamiento', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

print("💡 TODO: [Completa aquí] ¿Qué características tienen distribuciones muy diferentes?")

### 📏 Observación Importante: Escalas Diferentes

**TODO: [Completa aquí] ¿Por qué es un problema que las características tengan escalas diferentes?**

Las características tienen diferentes escalas:
- **age**: 29-77 años
- **trestbps**: 94-200 mm Hg
- **chol**: 126-564 mg/dl
- **thalach**: 71-202

**Solución**: Aplicaremos **Standard Scaling** (estandarización) para que todas las características tengan:
- Media = 0
- Desviación estándar = 1

**TODO: [Completa aquí] ¿Por qué esto ayuda a los modelos de Machine Learning?**

## 🔧 Paso 6: Construir el Pipeline de Preprocesamiento

Crearemos un pipeline que prepare los datos automáticamente.

In [0]:
# TODO: [Completa aquí] ¿Cuáles son las variables categóricas y cuáles numéricas?
cat_attr = ["sex", "cp", "fbs", "restecg", "exang", "slope"]
num_attr = ["age", "trestbps", "chol", "thalach", "oldpeak", "ca", "thal"]

print("📋 Clasificación de variables:")
print(f"   📊 Numéricas ({len(num_attr)}): {num_attr}")
print(f"   🏷️  Categóricas ({len(cat_attr)}): {cat_attr}")

# TODO: [Completa aquí] ¿Qué hace SimpleImputer y por qué usamos la mediana?
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # Rellenar valores nulos
    ("std_scaler", StandardScaler())                # Estandarizar
])

# TODO: [Completa aquí] ¿Qué es ColumnTransformer y por qué lo necesitamos?
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attr),      # Pipeline para numéricas
    ("cat", OneHotEncoder(), cat_attr)    # One-Hot Encoding para categóricas
])

print("\n✅ Pipeline de preprocesamiento creado")
print("\n💡 El pipeline:")
print("   1. Rellena valores nulos con la mediana (numéricas)")
print("   2. Estandariza variables numéricas (media=0, std=1)")
print("   3. Aplica One-Hot Encoding a variables categóricas")
print("\n💡 TODO: [Completa aquí] ¿Por qué One-Hot Encoding para categóricas?")

## 🎯 Paso 7: Preparar X e y

Separamos características (X) de la variable objetivo (y).

In [0]:
# TODO: [Completa aquí] ¿Por qué separamos X (características) de y (target)?
x_train = train_set.drop("target", axis=1)  # Características
y_train = train_set.target                   # Variable objetivo

print("=" * 70)
print("🎯 SEPARACIÓN DE CARACTERÍSTICAS Y TARGET")
print("=" * 70)
print(f"X_train shape: {x_train.shape} (características)")
print(f"y_train shape: {y_train.shape} (target)")
print("=" * 70)

In [0]:
# TODO: [Completa aquí] ¿Qué diferencia hay entre fit_transform y transform?
x_train_pr = full_pipeline.fit_transform(x_train)

print("🔧 Pipeline aplicado al conjunto de entrenamiento")
print(f"   Shape original: {x_train.shape}")
print(f"   Shape procesado: {x_train_pr.shape}")
print(f"\n💡 TODO: [Completa aquí] ¿Por qué cambió el número de columnas?")

## 🤖 Paso 8: Entrenamiento y Evaluación de Modelos

Entrenaremos dos modelos y compararemos sus resultados usando MLflow.

### 🔵 Modelo 1: SGD Classifier (Baseline)

**TODO: [Completa aquí] ¿Qué es SGD (Stochastic Gradient Descent) y cómo funciona?**

Empezaremos con un modelo simple como baseline (línea base) para tener una referencia.

In [0]:
# Activar autolog de MLflow
mlflow.sklearn.autolog()

# Iniciar experimento con MLflow
with mlflow.start_run(run_name="SGD Classifier - Baseline") as run:
    
    print("=" * 70)
    print("🚀 ENTRENANDO MODELO BASELINE: SGD CLASSIFIER")
    print("=" * 70)
    
    # TODO: [Completa aquí] ¿Qué es random_state y por qué lo usamos?
    sgd_clf = SGDClassifier(random_state=42)
    
    # TODO: [Completa aquí] ¿Qué es validación cruzada con 3 folds?
    print("\n🔄 Realizando validación cruzada (3-fold)...")
    scores = cross_val_score(sgd_clf, x_train_pr, y_train, cv=3, scoring="accuracy")
    
    print(f"\n📊 Accuracy por fold:")
    for i, score in enumerate(scores, 1):
        print(f"   Fold {i}: {score:.4f} ({score*100:.2f}%)")
    
    print(f"\n📈 Accuracy promedio: {scores.mean():.4f} ± {scores.std():.4f}")
    
    # Registrar métrica en MLflow
    mlflow.log_metric("cv_accuracy_mean", scores.mean())
    mlflow.log_metric("cv_accuracy_std", scores.std())
    
    print("\n✅ Modelo SGD entrenado con validación cruzada")
    print("=" * 70)

### 📊 Evaluación del Modelo SGD

In [0]:
# TODO: [Completa aquí] ¿Por qué usamos cross_val_predict en lugar de solo predecir?
print("🔮 Obteniendo predicciones con validación cruzada...")
preds = cross_val_predict(sgd_clf, x_train_pr, y_train, cv=3)
print(f"✅ {len(preds)} predicciones generadas")

In [0]:
# TODO: [Completa aquí] ¿Qué nos muestra la matriz de confusión?
print("📊 Generando matriz de confusión...\n")

cm = confusion_matrix(y_train, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Sin enfermedad', 'Con enfermedad'])
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues')
plt.title('Matriz de Confusión - SGD Classifier', fontsize=14, fontweight='bold')
plt.grid(False)
plt.show()

print(f"\n📋 Interpretación de la matriz:")
print(f"   TN (Verdaderos Negativos): {cm[0,0]} - Correctamente identificados como sanos")
print(f"   FP (Falsos Positivos): {cm[0,1]} - Sanos clasificados como enfermos")
print(f"   FN (Falsos Negativos): {cm[1,0]} - Enfermos clasificados como sanos ⚠️")
print(f"   TP (Verdaderos Positivos): {cm[1,1]} - Correctamente identificados como enfermos")
print(f"\n💡 TODO: [Completa aquí] ¿Qué tipo de error es más grave en medicina?")

In [0]:
# TODO: [Completa aquí] ¿Qué mide cada una de estas métricas?
print("=" * 70)
print("📊 MÉTRICAS DEL MODELO SGD")
print("=" * 70)

precision = precision_score(y_train, preds)
recall = recall_score(y_train, preds)
f1 = f1_score(y_train, preds)
roc_auc = roc_auc_score(y_train, preds)

print(f"🎯 Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"   → De los que predije como enfermos, el {precision*100:.1f}% realmente lo están")
print(f"\n❤️  Recall: {recall:.4f} ({recall*100:.2f}%)")
print(f"   → Detecté el {recall*100:.1f}% de todos los enfermos reales")
print(f"\n⚖️  F1 Score: {f1:.4f}")
print(f"   → Balance entre precision y recall")
print(f"\n📈 ROC-AUC Score: {roc_auc:.4f}")
print(f"   → Capacidad de discriminación del modelo")
print("=" * 70)

print("\n💡 TODO: [Completa aquí] ¿Qué métrica es más importante en medicina y por qué?")

### 🌲 Modelo 2: Random Forest Classifier

**TODO: [Completa aquí] ¿Qué es Random Forest y por qué suele funcionar mejor que modelos simples?**

Ahora probemos un modelo más potente y comparemos resultados.

In [0]:
# Continuar con MLflow (nuevo run para Random Forest)
with mlflow.start_run(run_name="Random Forest Classifier") as run:
    
    print("=" * 70)
    print("🚀 ENTRENANDO MODELO: RANDOM FOREST CLASSIFIER")
    print("=" * 70)
    
    # TODO: [Completa aquí] ¿Qué hiperparámetros podríamos ajustar en Random Forest?
    rf_clf = RandomForestClassifier(random_state=42)
    
    # TODO: [Completa aquí] ¿Por qué volvemos a hacer validación cruzada?
    print("\n🔄 Realizando validación cruzada (3-fold)...")
    rf_scores = cross_val_score(rf_clf, x_train_pr, y_train, cv=3, scoring="accuracy")
    
    print(f"\n📊 Accuracy por fold:")
    for i, score in enumerate(rf_scores, 1):
        print(f"   Fold {i}: {score:.4f} ({score*100:.2f}%)")
    
    print(f"\n📈 Accuracy promedio: {rf_scores.mean():.4f} ± {rf_scores.std():.4f}")
    
    # Obtener predicciones
    rf_preds = cross_val_predict(rf_clf, x_train_pr, y_train, cv=3)
    
    # Registrar en MLflow
    mlflow.log_metric("cv_accuracy_mean", rf_scores.mean())
    mlflow.log_metric("cv_accuracy_std", rf_scores.std())
    
    print("\n✅ Modelo Random Forest entrenado")
    print("=" * 70)

In [0]:
# TODO: [Completa aquí] ¿Esperamos que la matriz de confusión sea mejor? ¿Por qué?
print("📊 Matriz de Confusión - Random Forest\n")

cm_rf = confusion_matrix(y_train, rf_preds)
disp_rf = ConfusionMatrixDisplay(cm_rf, display_labels=['Sin enfermedad', 'Con enfermedad'])
fig, ax = plt.subplots(figsize=(8, 6))
disp_rf.plot(ax=ax, cmap='Greens')
plt.title('Matriz de Confusión - Random Forest', fontsize=14, fontweight='bold')
plt.grid(False)
plt.show()

print(f"\n📋 Interpretación:")
print(f"   TN: {cm_rf[0,0]} | FP: {cm_rf[0,1]}")
print(f"   FN: {cm_rf[1,0]} | TP: {cm_rf[1,1]}")
print(f"\n💡 TODO: [Completa aquí] ¿Mejoró respecto a SGD? ¿En qué?")

In [0]:
# TODO: [Completa aquí] ¿Cómo se comparan estas métricas con las de SGD?
print("=" * 70)
print("📊 MÉTRICAS DEL MODELO RANDOM FOREST")
print("=" * 70)

rf_precision = precision_score(y_train, rf_preds)
rf_recall = recall_score(y_train, rf_preds)
rf_f1 = f1_score(y_train, rf_preds)
rf_roc_auc = roc_auc_score(y_train, rf_preds)

print(f"🎯 Precision: {rf_precision:.4f} ({rf_precision*100:.2f}%)")
print(f"❤️  Recall: {rf_recall:.4f} ({rf_recall*100:.2f}%)")
print(f"⚖️  F1 Score: {rf_f1:.4f}")
print(f"📈 ROC-AUC Score: {rf_roc_auc:.4f}")
print("=" * 70)

# Comparación con SGD
print("\n📊 COMPARACIÓN SGD vs RANDOM FOREST")
print("=" * 70)
comparison = pd.DataFrame({
    'Métrica': ['Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'SGD': [precision, recall, f1, roc_auc],
    'Random Forest': [rf_precision, rf_recall, rf_f1, rf_roc_auc],
    'Diferencia': [
        rf_precision - precision,
        rf_recall - recall,
        rf_f1 - f1,
        rf_roc_auc - roc_auc
    ]
})
print(comparison.to_string(index=False))
print("=" * 70)

print("\n💡 TODO: [Completa aquí] ¿Qué modelo funciona mejor y por qué?")

## 🎯 Paso 9: Entrenamiento Final y Evaluación en Test

Ahora entrenaremos el modelo con **todo el conjunto de entrenamiento** y evaluaremos en el conjunto de prueba (que nunca ha visto).

In [0]:
# TODO: [Completa aquí] ¿Por qué ahora entrenamos con todos los datos de train?
print("🎯 Entrenamiento final con todo el conjunto de entrenamiento\n")

forest_clf = RandomForestClassifier(random_state=42)
forest_clf.fit(x_train_pr, y_train)

print("✅ Modelo Random Forest entrenado con", len(x_train_pr), "muestras")
print("📊 El modelo ahora tiene más datos para aprender patrones")

In [0]:
# TODO: [Completa aquí] ¿Por qué preparamos los datos de test de la misma manera?
x_test = test_set.drop("target", axis=1)
y_test = test_set.target

print("🧪 Conjunto de prueba preparado:")
print(f"   X_test: {x_test.shape}")
print(f"   y_test: {y_test.shape}")

In [0]:
# TODO: [Completa aquí] ¿Por qué usamos transform() y no fit_transform()?
x_test_pr = full_pipeline.transform(x_test)
final_preds = forest_clf.predict(x_test_pr)

print("🔮 Predicciones en conjunto de prueba generadas")
print(f"   Datos procesados: {x_test_pr.shape}")
print(f"   Predicciones: {len(final_preds)}")
print("\n💡 TODO: [Completa aquí] ¿Por qué NO debemos usar fit_transform en test?")

In [0]:
# TODO: [Completa aquí] ¿Qué significan estos resultados en el conjunto de prueba?
print("=" * 70)
print("🎉 RESULTADOS FINALES EN CONJUNTO DE PRUEBA")
print("=" * 70)

final_precision = precision_score(y_test, final_preds)
final_recall = recall_score(y_test, final_preds)
final_f1 = f1_score(y_test, final_preds)
final_roc_auc = roc_auc_score(y_test, final_preds)
final_accuracy = accuracy_score(y_test, final_preds)

print(f"🎯 Accuracy: {final_accuracy:.4f} ({final_accuracy*100:.2f}%)")
print(f"🎯 Precision: {final_precision:.4f} ({final_precision*100:.2f}%)")
print(f"❤️  Recall: {final_recall:.4f} ({final_recall*100:.2f}%)")
print(f"⚖️  F1 Score: {final_f1:.4f}")
print(f"📈 ROC-AUC: {final_roc_auc:.4f}")
print("=" * 70)

# Matriz de confusión final
print("\n📊 Matriz de Confusión Final:\n")
cm_final = confusion_matrix(y_test, final_preds)
disp_final = ConfusionMatrixDisplay(cm_final, display_labels=['Sin enfermedad', 'Con enfermedad'])
fig, ax = plt.subplots(figsize=(8, 6))
disp_final.plot(ax=ax, cmap='YlGnBu')
plt.title('Matriz de Confusión - Conjunto de Prueba', fontsize=14, fontweight='bold')
plt.grid(False)
plt.show()

# Reporte de clasificación completo
print("\n📋 REPORTE DE CLASIFICACIÓN DETALLADO:")
print("=" * 70)
print(classification_report(y_test, final_preds, 
                          target_names=['Sin enfermedad', 'Con enfermedad'],
                          digits=4))
print("=" * 70)

print("\n💡 TODO: [Completa aquí] ¿El modelo generaliza bien? ¿Cómo lo sabes?")

## 🎉 ¡Proyecto Completado!

### ✅ Lo que Hemos Logrado

1. ✅ **Explorado datos médicos reales** con análisis visual
2. ✅ **Construido un pipeline completo** de preprocesamiento
3. ✅ **Entrenado y comparado** 2 modelos (SGD y Random Forest)
4. ✅ **Evaluado con múltiples métricas** de clasificación
5. ✅ **Usado validación cruzada** para validación robusta
6. ✅ **Registrado experimentos** en MLflow
7. ✅ **Interpretado resultados** con matrices de confusión

### 📊 Conclusiones Clave

**TODO: [Completa aquí] Basándote en todos los resultados, escribe tus conclusiones:**

1. **¿Qué modelo funcionó mejor?**
   - [Tu respuesta aquí]

2. **¿Por qué crees que funcionó mejor?**
   - [Tu respuesta aquí]

3. **¿El modelo es suficientemente bueno para uso médico?**
   - [Tu respuesta aquí]

4. **¿Qué métrica es más importante en este caso?**
   - [Tu respuesta aquí]

5. **¿Qué mejorarías del modelo?**
   - [Tu respuesta aquí]

---

### 🎓 Conceptos Aprendidos

#### 📚 Machine Learning
- Clasificación binaria
- Validación cruzada
- Train/test split
- Overfitting vs generalización

#### 🔧 Preprocesamiento
- Normalización (StandardScaler)
- One-Hot Encoding
- Imputación de valores nulos
- Pipelines de transformación

#### 📊 Evaluación
- Accuracy, Precision, Recall, F1
- Matriz de confusión
- ROC-AUC
- Trade-offs entre métricas

#### 🤖 Modelos
- SGD Classifier
- Random Forest
- Comparación de modelos
- Selección de modelo

---

### 💡 Reflexiones Importantes

#### ⚕️ Sobre Medicina y IA

**TODO: [Completa aquí] Reflexiona sobre:**

1. **Falsos Negativos vs Falsos Positivos**
   - En medicina, ¿cuál es más grave y por qué?
   - [Tu respuesta aquí]

2. **Responsabilidad Ética**
   - ¿Puede un modelo de IA tomar decisiones médicas solo?
   - [Tu respuesta aquí]

3. **Interpretabilidad**
   - ¿Por qué es importante que los médicos entiendan cómo decide el modelo?
   - [Tu respuesta aquí]

---

### 🚀 Próximos Pasos

#### 🎯 Desafíos Adicionales

1. **Optimización de Hiperparámetros**
   ```python
   # TODO: Implementa GridSearchCV
   from sklearn.model_selection import GridSearchCV
   param_grid = {
       'n_estimators': [50, 100, 200],
       'max_depth': [5, 10, 15, None],
       'min_samples_split': [2, 5, 10]
   }
   ```

2. **Prueba Otros Modelos**
   - Gradient Boosting
   - XGBoost
   - SVM
   - Neural Networks

3. **Feature Engineering**
   - Crea nuevas características
   - Analiza importancia de características
   - Selecciona las más relevantes

4. **Análisis de Errores**
   - ¿Qué pacientes se clasifican mal?
   - ¿Hay patrones en los errores?
   - ¿Cómo mejorar?

5. **Curva ROC**
   - Implementa y visualiza la curva ROC
   - Encuentra el threshold óptimo
   - Compara AUC de diferentes modelos

---

### 📖 Recursos para Seguir Aprendiendo

- **Scikit-learn**: [Classification Metrics](https://scikit-learn.org/stable/modules/model_evaluation.html)
- **MLflow**: [Tracking Documentation](https://mlflow.org/docs/latest/tracking.html)
- **Kaggle**: Notebooks sobre Heart Disease
- **Papers**: Sobre IA en diagnóstico médico

---

### 🌟 ¡Felicidades!

Has completado un proyecto completo de Machine Learning en el dominio médico. Las habilidades que has desarrollado son aplicables a muchos otros problemas de clasificación.

**Puntos clave para recordar:**
- ✅ Siempre explora tus datos primero
- ✅ Usa validación cruzada para evaluar
- ✅ Compara múltiples modelos
- ✅ Elige métricas según el problema
- ✅ En medicina, prioriza recall sobre precision
- ✅ Documenta todo con MLflow
- ✅ Piensa en las implicaciones éticas

**¡Sigue practicando y construyendo proyectos! 🚀❤️**

In [ ]:
# 🎨 CÓDIGO DE EJEMPLO PARA LOS DESAFÍOS
# Descomenta y completa para explorar más

# ========================================
# DESAFÍO 1: Grid Search
# ========================================
# from sklearn.model_selection import GridSearchCV
# 
# param_grid = {
#     'n_estimators': [50, 100, 200],
#     'max_depth': [5, 10, 15, None],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4]
# }
# 
# # TODO: [Completa aquí] ¿Qué hace GridSearchCV?
# mlflow.sklearn.autolog()
# with mlflow.start_run(run_name="Grid Search - Random Forest"):
#     grid_search = GridSearchCV(
#         RandomForestClassifier(random_state=42),
#         param_grid,
#         cv=5,
#         scoring='recall',  # Priorizamos recall en medicina
#         verbose=1,
#         n_jobs=-1
#     )
#     grid_search.fit(x_train_pr, y_train)
#     
#     print(f"Mejores parámetros: {grid_search.best_params_}")
#     print(f"Mejor recall (CV): {grid_search.best_score_:.4f}")

# ========================================
# DESAFÍO 2: Curva ROC
# ========================================
# from sklearn.metrics import roc_curve, auc
# 
# # TODO: [Completa aquí] ¿Qué es la curva ROC y qué nos dice?
# # Obtener probabilidades
# y_proba = forest_clf.predict_proba(x_test_pr)[:, 1]
# fpr, tpr, thresholds = roc_curve(y_test, y_proba)
# roc_auc_curve = auc(fpr, tpr)
# 
# plt.figure(figsize=(10, 8))
# plt.plot(fpr, tpr, color='darkorange', lw=2, 
#          label=f'ROC curve (AUC = {roc_auc_curve:.2f})')
# plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
# plt.xlim([0.0, 1.0])
# plt.ylim([0.0, 1.05])
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate (Recall)')
# plt.title('Receiver Operating Characteristic (ROC) Curve')
# plt.legend(loc="lower right")
# plt.grid(True, alpha=0.3)
# plt.show()

# ========================================
# DESAFÍO 3: Importancia de Características
# ========================================
# # TODO: [Completa aquí] ¿Qué características son más importantes?
# feature_names = num_attr + list(full_pipeline.named_transformers_['cat'].get_feature_names_out(cat_attr))
# feature_importance = pd.DataFrame({
#     'feature': feature_names,
#     'importance': forest_clf.feature_importances_
# }).sort_values('importance', ascending=False)
# 
# plt.figure(figsize=(10, 8))
# plt.barh(feature_importance.head(15)['feature'], 
#          feature_importance.head(15)['importance'])
# plt.xlabel('Importancia')
# plt.title('Top 15 Características Más Importantes')
# plt.gca().invert_yaxis()
# plt.tight_layout()
# plt.show()

# ========================================
# DESAFÍO 4: Comparar Más Modelos
# ========================================
# from sklearn.ensemble import GradientBoostingClassifier
# from sklearn.svm import SVC
# from sklearn.linear_model import LogisticRegression
# 
# modelos = {
#     'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
#     'SVM': SVC(kernel='rbf', probability=True, random_state=42),
#     'Gradient Boosting': GradientBoostingClassifier(random_state=42)
# }
# 
# # TODO: [Completa aquí] ¿Cuál modelo funciona mejor?
# resultados = []
# mlflow.sklearn.autolog()
# 
# for nombre, modelo in modelos.items():
#     with mlflow.start_run(run_name=f"{nombre} - Heart Disease"):
#         modelo.fit(x_train_pr, y_train)
#         preds = modelo.predict(x_test_pr)
#         
#         acc = accuracy_score(y_test, preds)
#         prec = precision_score(y_test, preds)
#         rec = recall_score(y_test, preds)
#         f1_s = f1_score(y_test, preds)
#         
#         resultados.append({
#             'Modelo': nombre,
#             'Accuracy': acc,
#             'Precision': prec,
#             'Recall': rec,
#             'F1-Score': f1_s
#         })
# 
# df_resultados = pd.DataFrame(resultados).sort_values('Recall', ascending=False)
# print("\n📊 Ranking de Modelos (ordenado por Recall):")
# print(df_resultados.to_string(index=False))

# ========================================
# DESAFÍO 5: Análisis de Errores
# ========================================
# # TODO: [Completa aquí] ¿Qué pacientes son difíciles de clasificar?
# # Encontrar errores
# errors_mask = final_preds != y_test
# errors_df = test_set[errors_mask].copy()
# errors_df['prediccion'] = final_preds[errors_mask]
# 
# print(f"\nTotal de errores: {errors_mask.sum()}")
# print(f"\nPacientes mal clasificados:")
# print(errors_df[['age', 'sex', 'cp', 'trestbps', 'chol', 'target', 'prediccion']])
# 
# # Analizar características de los errores
# print("\n¿Los errores tienen características en común?")
# print(errors_df.describe())

print("💡 Descomenta el código del desafío que quieras explorar")